In [ ]:
import pandas as pd
import numpy as np
from tools import sherlock

In [ ]:
df = pd.read_csv(r"..\data\processed\2BLONDEL_consolide.csv", sep=';', decimal=',', parse_dates=['date_écriture'])

In [ ]:
sherlock(df)

In [ ]:
df.head()

---
# CA

code_compte == '70600000' -> loyers

## CA Global

In [ ]:
loyers = df.query("code_compte == '70600000'")

In [ ]:
credit = loyers['crédit_euro'].sum()
debit = loyers['débit_euro'].sum()
credit - debit

## CA Annuel

In [ ]:
ca_annuel = loyers.groupby('année')[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro').reset_index()
ca_annuel

## CA Mensuel

In [ ]:
ca_mensuel = loyers.groupby(loyers['date_écriture'].dt.to_period('M'))[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro').reset_index()
ca_mensuel

## CA par Plateforme

In [ ]:
conds = [
    loyers['libellé_écriture'].str.contains('BNB|FACTURE HOTE', case=False),
    loyers['libellé_écriture'].str.contains('BOOKING', case=False),
    ]
choices = ['AIRBNB', 'BOOKING']
loyers['canal'] = np.select(conds, choices, default='DIRECT')

### CA par PTF

In [ ]:
loyers.groupby('canal')[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro').reset_index()

### CA par PTF annuel

In [ ]:
loyers.groupby(['année', 'canal'])[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro').reset_index()

### CA par PTF mensuel

In [ ]:
loyers.groupby([loyers['date_écriture'].dt.to_period('M'), 'canal'])[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro').reset_index()

---
# Charges

In [ ]:
charges = df[df['code_compte'].str.startswith('6') & (df['débit_euro'] > 0)]

## Charges annuelles

In [ ]:
charges.groupby('année')['débit_euro'].sum().reset_index()

## Details par Categories

In [ ]:
charges.groupby(['code_compte', 'intitulé_compte'])['débit_euro'].sum().reset_index()

### Focus sur '60630000'

In [ ]:
charges.query("code_compte == '60630000'").sort_values('débit_euro', ascending=False).head(10)

### Focus sur '66116000'

In [ ]:
charges.query("code_compte == '66116000'").sort_values('débit_euro', ascending=False).head(10)

### Focus sur '62270000'

In [ ]:
charges.query("code_compte == '62270000'").sort_values('débit_euro', ascending=False).head(10)

### Focus sur '62510000'

In [ ]:
charges.query("code_compte == '62510000'").sort_values('débit_euro', ascending=False).head(10)

## Charges Mensuelles

In [ ]:
charges.groupby(charges['date_écriture'].dt.to_period('M'))['débit_euro'].sum().reset_index()

### Focus sur 12/2025

In [ ]:
charges[(charges['date_écriture'].dt.to_period('M') == '2024-12')]

## Mensuel de la Categorie "Entretien et petit équipement"

In [ ]:
charges.query("code_compte == '60630000'").groupby(charges['date_écriture'].dt.to_period('M'))['débit_euro'].sum().reset_index()

### Focus sur 04/2025

In [ ]:
charges[(charges['code_compte'] == '60630000') & (charges['date_écriture'].dt.to_period('M') == '2025-04')]

## Fixes vs Variables

In [ ]:
fixes = ['61610000', '66116000', '62620000', '62780000', '63512000', '62810000', '68112000']
semi_fixes = ['60630000', '61520000', '61560000']
variables = ['60611000', '60614000', '62220000', '62260000', '62270000', '62310000', '62340000', '62510000', '62610000', '65800000']

conds = [
    charges['code_compte'].isin(fixes),
    charges['code_compte'].isin(semi_fixes),
    charges['code_compte'].isin(variables)
]

choices = ['fixe', 'semi_fixe', 'variable']

charges['categorie'] = np.select(conds, choices, default='autre')

In [ ]:
# Sommes par Catégories

charges.groupby('categorie')['débit_euro'].sum()

---
# Cashflow

## Mensuel

In [ ]:
recette = loyers.groupby(loyers['date_écriture'].dt.to_period('M'))[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro').reset_index()
recette = recette.drop(columns=['crédit_euro', 'débit_euro'])
depense = charges.groupby(charges['date_écriture'].dt.to_period('M'))['débit_euro'].sum().reset_index()
depense = depense.rename(columns={'débit_euro':'charges'})

# Merge
cashflow = recette.merge(depense, how='outer', on='date_écriture')
cashflow = cashflow.fillna(0)

# Cashflow
cashflow['cashflow'] = cashflow['ca_net'] - cashflow['charges']
cashflow

## Annuel

In [ ]:
recette = loyers.groupby(loyers['date_écriture'].dt.to_period('Y'))[['crédit_euro', 'débit_euro']].sum().eval('ca_net = crédit_euro - débit_euro').reset_index()
recette = recette.drop(columns=['crédit_euro', 'débit_euro'])
depense = charges.groupby(charges['date_écriture'].dt.to_period('Y'))['débit_euro'].sum().reset_index()
depense = depense.rename(columns={'débit_euro':'charges'})

# Merge
cashflow = recette.merge(depense, how='outer', on='date_écriture')
cashflow = cashflow.fillna(0)

# Cashflow
cashflow['cashflow'] = cashflow['ca_net'] - cashflow['charges']
cashflow

---
---
# Correction anomalie juillet 2025

Juillet 2025 présente une chute anormale documentée (-82% consultations Airbnb, -94% réservations vs juillet 2024). Le CA comptable brut de 17 362€ est donc sous-estimé. On corrige en remplaçant le CA juillet 2025 réel par la moyenne des mois adjacents (avril à septembre 2025), plus représentative d'un juillet normal.

In [ ]:
ca_mensuel

In [ ]:
mois_ref = ['2025-04', '2025-05', '2025-06', '2025-08', '2025-09']
mask = ca_mensuel['date_écriture'].astype(str).isin(mois_ref)

ca_cible = ca_mensuel[mask]['ca_net'].mean()
ca_juil25 = ca_mensuel[ca_mensuel['date_écriture'].astype(str) == '2025-07']['ca_net'].item()
manque = ca_cible - ca_juil25

ca2025_corrige = ca_annuel.query("année == 2025")['ca_net'].item() + manque

print(f"CA Juillet 2025 = {ca_juil25:.2f} €")
print(f"Moyenne saison haute = {ca_cible:.2f} €")
print(f"Manque à gagner = {manque:.2f} €")
print(f"CA 2025 corrigé = {ca2025_corrige:.2f} €")

---
# RevPAR 2025 — Estimation comptable

Le RevPAR (Revenue Per Available Room/Night) mesure le revenu généré par jour disponible, qu'il soit vendu ou non. Contrairement à l'ADR (prix moyen par nuit vendue), il intègre le taux d'occupation — c'est l'indicateur de performance globale d'un hébergement.

Le RevPAR 2024 réel a été calculé à partir des données de taxe de séjour (nuits occupées + jours disponibles réels par mois) → voir notebook 7.

Pour 2025, la taxe de séjour n'est pas disponible. Le RevPAR est donc estimé à partir du CA comptable annuel divisé par une hypothèse de 340 jours disponibles (25 jours réservés maintenance/usage propriétaires). C'est une approximation annuelle — le détail mensuel n'est pas fiable à cause du décalage entre date de séjour et date d'encaissement.

In [ ]:
ca2025 = ca_annuel.query("année == 2025")['ca_net']
revpar2025 = ca2025.item() / 340
revpar2025_corrige = ca2025_corrige / 340
print(f"RevPAR 2025 = {revpar2025:.2f} €")
print(f"RevPAR 2025 (sur CA corrigé) = {revpar2025_corrige:.2f} €")